# 03. B2B Propensity Scoring, Channel Classification & Dual Export

Validates SCIAN chronotype imputation, digital maturity audit, B2B lead propensity scoring, activation channel recommendation, ethical DuckDuckGo enrichment, and clean-room dual export.

**Philosophy:** Self-contained functions annotated with `# MIGRATES TO:` for extraction into `src/mx_osint/enrichment/`, `src/mx_osint/activation/`, and `src/mx_osint/reporting/`.

In [ ]:
# Cell 3.1: Load Resolved Data + Data Contracts
import json
from pathlib import Path
import polars as pl

resolved_path = Path("data/processed/entities_resolved.parquet")
if resolved_path.exists():
    df = pl.read_parquet(resolved_path)
    print(f"Loaded resolved entities: {len(df)}")
else:
    fixture_path = Path("fixtures/synthetic_denue_sample.parquet")
    print(f"Fallback to synthetic fixture: {fixture_path}")
    df = pl.read_parquet(fixture_path)
    if "distance_km" not in df.columns:
        df = df.with_columns(pl.lit(1.2).alias("distance_km"))

with open("data_contracts/scian_chronotype_mapping.json", "r", encoding="utf-8") as f:
    chronotype_mapping = json.load(f)

with open("data_contracts/scian_b2b_propensity.json", "r", encoding="utf-8") as f:
    b2b_propensity_weights = json.load(f)

print(f"Loaded contracts: {len(chronotype_mapping)} chronotypes, {len(b2b_propensity_weights)} propensity rules.")

In [ ]:
# MIGRATES TO: src/mx_osint/enrichment/chronotype_imputer.py::impute_chronotype
from typing import Dict, Any
import polars as pl

def impute_chronotype(df: pl.DataFrame, mapping: Dict[str, Any]) -> pl.DataFrame:
    """Assigns operating chronotype based on SCIAN activity prefix (2-digit sector code)."""
    chronotypes = []
    for code in df["codigo_act"].to_list():
        code_str = str(code).strip()
        prefix2 = code_str[:2]
        if prefix2 in mapping:
            chronotypes.append(mapping[prefix2].get("chronotype", "Unknown"))
        else:
            chronotypes.append("Office_Standard_8AM")
    return df.with_columns(pl.Series("chronotype", chronotypes))

In [ ]:
# MIGRATES TO: src/mx_osint/enrichment/digital_maturity.py::classify_digital_maturity
import re
import polars as pl

FREE_EMAIL_REGEX = re.compile(r"@(gmail|hotmail|yahoo|outlook|live|prodigy\.net\.mx|icloud)\.", re.IGNORECASE)

def classify_digital_maturity(df: pl.DataFrame) -> pl.DataFrame:
    """Audits presence of email, website, and phone; categorizes corporate vs free email domains."""
    emails = df["correoelec"].to_list() if "correoelec" in df.columns else [""] * len(df)
    websites = df["www"].to_list() if "www" in df.columns else [""] * len(df)
    phones = df["telefono"].to_list() if "telefono" in df.columns else [""] * len(df)

    has_email = []
    email_type = []
    has_website = []
    has_phone = []
    maturity_scores = []
    digital_gaps = []

    for em, ws, ph in zip(emails, websites, phones):
        em_str = str(em).strip() if em else ""
        ws_str = str(ws).strip() if ws else ""
        ph_str = str(ph).strip() if ph else ""

        e_present = bool(em_str and em_str.lower() not in ["none", "null", "no tiene"])
        w_present = bool(ws_str and ws_str.lower() not in ["none", "null", "no tiene"])
        p_present = bool(ph_str and ph_str.lower() not in ["none", "null", "no tiene"])

        if e_present:
            e_type = "free_email" if FREE_EMAIL_REGEX.search(em_str) else "corporate_domain"
        else:
            e_type = "none"

        # Composite score 0 to 3
        score = 0
        if e_type == "corporate_domain":
            score += 2
        elif e_type == "free_email":
            score += 1
        if w_present:
            score += 1

        has_email.append(e_present)
        email_type.append(e_type)
        has_website.append(w_present)
        has_phone.append(p_present)
        maturity_scores.append(min(score, 3))
        digital_gaps.append(not e_present and not w_present)

    return df.with_columns([
        pl.Series("has_email", has_email),
        pl.Series("email_type", email_type),
        pl.Series("has_website", has_website),
        pl.Series("has_phone", has_phone),
        pl.Series("digital_maturity_score", maturity_scores),
        pl.Series("digital_gap_flag", digital_gaps)
    ])

In [ ]:
# MIGRATES TO: src/mx_osint/enrichment/b2b_propensity.py::score_b2b_propensity
from typing import Dict, Any
import polars as pl

STRATA_MULTIPLIERS = {
    "0 a 5 personas": 0.4,
    "6 a 10 personas": 0.6,
    "11 a 30 personas": 0.8,
    "31 a 50 personas": 1.0,
    "51 a 100 personas": 1.2,
    "101 a 250 personas": 1.4,
    "251 y más personas": 1.6
}

def score_b2b_propensity(df: pl.DataFrame, scian_weights: Dict[str, Any]) -> pl.DataFrame:
    """Calculates composite B2B lead propensity score (0-100) based on sector, strata, proximity, and maturity."""
    scores = []
    codes = df["codigo_act"].to_list()
    stratas = df["per_ocu"].to_list()
    dists = df["distance_km"].to_list() if "distance_km" in df.columns else [1.5] * len(df)
    chronos = df["chronotype"].to_list() if "chronotype" in df.columns else ["Office_Standard_8AM"] * len(df)
    email_types = df["email_type"].to_list() if "email_type" in df.columns else ["none"] * len(df)

    for code, strata, dist, chrono, e_type in zip(codes, stratas, dists, chronos, email_types):
        p2 = str(code)[:2]
        base_weight = scian_weights.get(p2, {}).get("weight", 50.0)
        strata_mult = STRATA_MULTIPLIERS.get(strata, 0.7)
        
        # Distance decay (closer gets higher score)
        decay = max(0.5, 1.0 - (dist / 10.0))
        
        # Chronotype bonuses
        chrono_bonus = 20.0 if chrono == "Early_Bird_6AM" else (15.0 if chrono == "Office_Standard_8AM" else 5.0)
        
        # Digital maturity bonus
        digital_bonus = 10.0 if e_type == "corporate_domain" else 0.0
        
        raw_score = (base_weight * 0.5 * strata_mult * decay) + chrono_bonus + digital_bonus
        final_score = round(min(100.0, max(0.0, raw_score)), 1)
        scores.append(final_score)

    return df.with_columns(pl.Series("b2b_score", scores)).sort("b2b_score", descending=True)

In [ ]:
# MIGRATES TO: src/mx_osint/activation/channel_classifier.py::classify_channel
from typing import Dict, Any
import polars as pl

def classify_channel_for_record(row: Dict[str, Any]) -> str:
    """Determines optimal outreach channel for a single establishment record."""
    if row.get("email_type") == "corporate_domain":
        return "Email_RH_Compras"
    if row.get("has_website"):
        return "Web_Form_Scraping"
    scian_code = str(row.get("codigo_act", ""))
    strata = str(row.get("per_ocu", ""))
    if scian_code[:2] in ["31", "32", "33"] and strata in ["101 a 250 personas", "251 y más personas"]:
        return "LinkedIn_Corporativo"
    if row.get("has_phone"):
        return "Recepcion_Directa"
    return "Digital_Gap_Needs_DuckDuckGo"

def classify_channel(df: pl.DataFrame) -> pl.DataFrame:
    """Applies outreach channel classification across the dataframe."""
    rows = df.to_dicts()
    channels = [classify_channel_for_record(r) for r in rows]
    return df.with_columns(pl.Series("channel", channels))

In [ ]:
# MIGRATES TO: src/mx_osint/activation/channel_classifier.py::discover_channel_ddg
import time
from typing import Dict, Any
from loguru import logger

def discover_channel_ddg(entity_name: str, state: str = "Guanajuato", max_results: int = 3, sleep_sec: float = 2.0) -> Dict[str, Any]:
    """
    Ethical channel discovery via DuckDuckGo.
    WARNING: Only run on top-N prioritized prospects. Respect robots.txt. Never exceed 30 queries/minute.
    """
    time.sleep(sleep_sec)
    query = f'"{entity_name}" "{state}" contacto OR recursos-humanos OR recepcion'
    logger.info(f"Ethical DDG query: {query}")
    # Lazy import to avoid hard network dependency during unit tests
    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
            if results:
                return {"status": "found", "top_url": results[0].get("href"), "snippet": results[0].get("body")}
    except Exception as e:
        logger.warning(f"DDG lookup error for '{entity_name}': {e}")
    return {"status": "not_found", "top_url": None, "snippet": None}

In [ ]:
# MIGRATES TO: src/mx_osint/reporting/prospect_report.py::export_dual
import json
from pathlib import Path
from typing import Optional
import polars as pl
from loguru import logger

def export_dual(df: pl.DataFrame, private_path: Path, public_path: Optional[Path] = None) -> None:
    """
    Strict Clean-Room dual export:
    - Private: Full lead records (name, contact, score, channel). Gitignored.
    - Public: Aggregated statistics by SCIAN sector and municipality. Zero PII, safe for public git.
    """
    # 1. Private export
    private_path.parent.mkdir(parents=True, exist_ok=True)
    top30 = df.head(30)
    top30.write_csv(private_path)
    logger.info(f"Saved private top-30 prospect report to {private_path}")
    
    # 2. Public aggregated export
    if public_path:
        public_path.parent.mkdir(parents=True, exist_ok=True)
        agg_df = (
            df.with_columns(pl.col("codigo_act").str.slice(0, 2).alias("sector_scian"))
            .group_by(["sector_scian", "mun"])
            .agg([
                pl.len().alias("total_establishments"),
                pl.col("has_email").mean().alias("pct_email"),
                pl.col("has_website").mean().alias("pct_website"),
                pl.col("digital_maturity_score").mean().alias("avg_digital_maturity"),
                pl.col("b2b_score").mean().alias("avg_b2b_propensity")
            ])
        )
        records = agg_df.to_dicts()
        with open(public_path, "w", encoding="utf-8") as f:
            json.dump({"meta": {"generator": "mexico-commercial-osint", "version": "0.1.0"}, "aggregates": records}, f, indent=2)
        logger.info(f"Saved public aggregate index to {public_path}")

In [ ]:
# Cell 3.8: Pipeline Execution & Map Generation
from pathlib import Path
import polars as pl

# Apply transformations
enriched_df = impute_chronotype(df, chronotype_mapping)
enriched_df = classify_digital_maturity(enriched_df)
scored_df = score_b2b_propensity(enriched_df, b2b_propensity_weights)
final_df = classify_channel(scored_df)

# Export private report
priv_csv = Path("data/private/top_30_b2b_prospects.csv")
export_dual(final_df, priv_csv)
print(f"Successfully generated {priv_csv} with {min(30, len(final_df))} prospects.")

# Generate interactive Folium map (Private)
try:
    import folium
    top30 = final_df.head(30)
    m = folium.Map(location=[top30["lat"][0], top30["lon"][0]], zoom_start=14)
    for r in top30.to_dicts():
        popup = f"<b>{r['nombre_estab']}</b><br>Score: {r['b2b_score']}<br>Channel: {r['channel']}"
        folium.Marker([r["lat"], r["lon"]], popup=popup, tooltip=r["nombre_estab"]).add_to(m)
    map_path = Path("data/private/prospects_map.html")
    m.save(str(map_path))
    print(f"Saved interactive prospect map to {map_path}")
except Exception as e:
    print(f"Folium map skipped: {e}")